# Penalty Kick Prediction - Project 3, Question 4

All Q4 code lives in this notebook. Q1-Q3 have their own `Q1to3Solution.ipynb`. Current scope is Phase 0: setup, reconnaissance, metric and submission checks. No training or competition predictions yet. Requirements come from the course PDF; see `AGENTS.md` for current findings and access limitations.

## Setup

Run cells in order with the project-local Python kernel. Only standard libraries and installed packages are imported; no project `.py` files are required.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile

import pandas as pd

ROOT = Path.cwd().resolve()
assert (ROOT / "AGENTS.md").exists(), "Set ROOT to the project directory"
DATA_DIR = ROOT / "Data"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
SEED = 6643
COMPETITION = "penalty-kick-prediction-cs-gy-6643"
print("Project:", ROOT.name, "| seed:", SEED)

Project: Project 3 | seed: 6643


## Optional official data download

Configure Kaggle credentials privately first; never put a token in a notebook cell. The download is disabled by default. Set `RUN_DOWNLOAD = True` to list the official files first. After confirming access, set `LIST_ONLY = False` to download. Existing inputs are never overwritten; this code never uploads a submission.

In [2]:
def extract_archive(archive, destination):
    """Preflight all members before writing into a fresh destination."""
    destination = Path(destination)
    if destination.exists():
        raise FileExistsError(f"Refusing to overwrite {destination}")
    with zipfile.ZipFile(archive) as zipped:
        members = zipped.infolist()
        names = [m.filename for m in members]
        if len(names) != len(set(names)):
            raise ValueError("Duplicate paths in archive")
        for member in members:
            path = Path(member.filename)
            mode = (member.external_attr >> 16) & 0o170000
            if path.is_absolute() or ".." in path.parts or "\\" in member.filename or mode == 0o120000:
                raise ValueError("Unsafe archive member")
        bad = zipped.testzip()
        if bad:
            raise ValueError(f"Corrupt archive member: {bad}")
        destination.mkdir(parents=True)
        try:
            zipped.extractall(destination)
        except BaseException:
            shutil.rmtree(destination)  # Only this new, incomplete extraction.
            raise


def download_competition(list_only=True):
    environment = os.environ.copy()
    environment.setdefault("KAGGLE_CONFIG_DIR", str(ROOT / ".kaggle"))
    Path(environment["KAGGLE_CONFIG_DIR"]).mkdir(mode=0o700, parents=True, exist_ok=True)
    executable = Path(sys.executable).parent / "kaggle"
    if not executable.is_file():
        raise RuntimeError("Install requirements-setup.txt into the notebook environment first")
    subprocess.run([str(executable), "competitions", "files", "-c", COMPETITION],
                   env=environment, cwd=ROOT, check=True)
    if list_only:
        return
    DATA_DIR.mkdir(exist_ok=True)
    destination = DATA_DIR / COMPETITION
    if destination.exists():
        raise FileExistsError(f"Raw data is immutable; refusing to overwrite {destination}")
    with tempfile.TemporaryDirectory(prefix=".download-", dir=DATA_DIR) as temp:
        staging = Path(temp)
        subprocess.run([str(executable), "competitions", "download", "-c", COMPETITION,
                        "-p", str(staging)], env=environment, cwd=ROOT, check=True)
        archives = list(staging.glob("*.zip"))
        if len(archives) != 1:
            raise ValueError("Expected one competition ZIP; inspect the official file listing")
        expanded = staging / "expanded"
        extract_archive(archives[0], expanded)
        expanded.rename(destination)
    print("Downloaded official files to", destination)
    return destination

In [3]:
RUN_DOWNLOAD = False
LIST_ONLY = True

if RUN_DOWNLOAD:
    download_competition(list_only=LIST_ONLY)
else:
    print("Download disabled. Kaggle authentication and competition access are still required.")

Download disabled. Kaggle authentication and competition access are still required.


## Official labels, metric, and submission checks

Source: course PDF pp. 8-12. Directions use the kicker's perspective. Scoring is 60% kick accuracy, 25% keeper accuracy, and 15% outcome accuracy. The formula below is reconstructed from the PDF; the live official scorer is unavailable. Submission IDs are read only from actual official files. Requiring template row order is a conservative local convention.

In [4]:
COLUMNS = ("clip", "kick_direction", "goalkeeper_direction", "goal_outcome")
LABELS = {
    "kick_direction": ("left", "middle", "right"),
    "goalkeeper_direction": ("left", "stands in the middle", "right"),
    "goal_outcome": ("goal", "no_goal"),
}
WEIGHTS = {"kick_direction": 0.60, "goalkeeper_direction": 0.25, "goal_outcome": 0.15}
EXPECTED_PRIVATE_ROWS = 900

In [5]:
def read_csv(path):
    """Keep identifiers and labels as exact strings, including leading zeros."""
    with Path(path).open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.reader(stream)
        try:
            columns = next(reader)
        except StopIteration:
            raise ValueError(f"Empty CSV: {path}") from None
        if len(columns) != len(set(columns)):
            raise ValueError(f"Duplicate column names: {path}")
        rows = []
        for number, values in enumerate(reader, 2):
            if len(values) != len(columns):
                raise ValueError(f"Wrong field count in {path}, line {number}")
            rows.append(dict(zip(columns, values)))
    return columns, rows


def identifiers(rows, description):
    ids = [row["clip"] for row in rows]
    if not ids or any(not x or x != x.strip() or not x.endswith(".mp4") for x in ids):
        raise ValueError(f"{description}: clip IDs must be nonempty exact .mp4 strings")
    if len(ids) != len(set(ids)):
        raise ValueError(f"{description}: duplicate clip IDs")
    return ids


def validate_submission(predictions, template, metadata, *, expected_rows=EXPECTED_PRIVATE_ROWS,
                        require_order=True):
    """Strict template order is a conservative local policy, not a claimed Kaggle rule."""
    pred_columns, pred = read_csv(predictions)
    template_columns, official = read_csv(template)
    metadata_columns, private = read_csv(metadata)
    if tuple(template_columns) != COLUMNS:
        raise ValueError("Official template differs from PDF contract; reconcile documentation first")
    if pred_columns != template_columns:
        raise ValueError(f"Columns must exactly match official template: {template_columns}")
    if "clip" not in metadata_columns:
        raise ValueError("Private metadata is missing clip")
    if any(name in metadata_columns for name in (*LABELS, "outcome_detail")):
        raise ValueError("Private metadata unexpectedly contains label fields; investigate provenance")
    pred_ids = identifiers(pred, "Predictions")
    template_ids = identifiers(official, "Template")
    metadata_ids = identifiers(private, "Metadata")
    if any(len(x) != expected_rows for x in (pred, official, private)):
        raise ValueError(f"Predictions, template, and metadata must each have {expected_rows} rows")
    if set(template_ids) != set(metadata_ids):
        raise ValueError("Official template and private metadata IDs do not correspond")
    if set(pred_ids) != set(metadata_ids):
        raise ValueError("Prediction IDs do not correspond exactly to private metadata")
    if require_order and pred_ids != template_ids:
        raise ValueError("Prediction order must match template (conservative local policy)")
    for index, row in enumerate(pred, 2):
        for target, allowed in LABELS.items():
            if row[target] not in allowed:
                raise ValueError(f"Line {index}: {target} must be one of {allowed}; got {row[target]!r}")
    return {"valid": True, "rows": len(pred), "columns": pred_columns,
            "template_order_required_locally": require_order}


def weighted_accuracy(truth, predictions):
    """Align by clip, reject duplicates and unknown labels, return component metrics."""
    def indexed(rows):
        result = {}
        for row in rows:
            clip = row["clip"]
            if not clip or clip in result:
                raise ValueError("Missing or duplicate clip")
            for target, allowed in LABELS.items():
                if row[target] not in allowed:
                    raise ValueError(f"Invalid {target}: {row[target]!r}")
            result[clip] = row
        return result

    actual, predicted = indexed(truth), indexed(predictions)
    if not actual or actual.keys() != predicted.keys():
        raise ValueError("Truth and prediction IDs must match and be nonempty")
    accuracies = {
        target: sum(actual[c][target] == predicted[c][target] for c in actual) / len(actual)
        for target in LABELS
    }
    return {**accuracies, "score": sum(WEIGHTS[t] * accuracies[t] for t in LABELS)}

## Read-only dataset audit

Inspect actual CSVs and videos once downloaded. Full video decoding and hashing are optional because they can be expensive. Public-only previews are saved under `outputs/figures/`. An unavailable dataset is reported as a blocker, not inferred to contain zero samples.

In [6]:
REQUIRED = ("public_ground_truth.csv", "private_metadata.csv",
            "public_position_clicks.csv", "submission_template.csv")
METADATA = ("clip", "source_id", "preferred_foot", "gender", "resolution", "fps")


def checksum(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def audit_data(raw_dir, *, decode_videos=False, hash_videos=False):
    raw_dir = Path(raw_dir)
    files = sorted(p for p in raw_dir.rglob("*") if p.is_file() and p.name != ".gitkeep")
    report = {"raw_dir": str(raw_dir), "status": "available" if files else "blocked_no_data",
              "inventory": [], "tables": {}, "issues": [], "video_checks": [],
              "decode_videos_requested": decode_videos, "hash_videos_requested": hash_videos}
    for path in files:
        report["inventory"].append({"path": str(path.relative_to(raw_dir)),
                                    "bytes": path.stat().st_size})
    tables = {}
    for name in REQUIRED:
        matches = [p for p in files if p.name == name]
        if len(matches) != 1:
            report["issues"].append(f"{name}: expected one file, found {len(matches)}")
            continue
        path = matches[0]
        try:
            columns, rows = read_csv(path)
        except (ValueError, UnicodeError) as exc:
            report["issues"].append(str(exc))
            continue
        tables[name] = rows
        ids = [r.get("clip", "") for r in rows]
        counts = Counter(ids)
        report["tables"][name] = {
            "path": str(path.relative_to(raw_dir)), "sha256": checksum(path),
            "columns": columns, "rows": len(rows),
            "missing_by_column": {c: sum(not r[c].strip() for r in rows) for c in columns},
            "duplicate_clip_ids": [c for c,n in counts.items() if n > 1],
            "duplicate_rows": len(rows) - len({tuple(r.values()) for r in rows}),
        }
        if "clip" not in columns:
            report["issues"].append(f"{name}: missing clip column")
        if any(n > 1 for n in counts.values()):
            report["issues"].append(f"{name}: duplicate clip IDs")
        if name in ("public_ground_truth.csv", "private_metadata.csv"):
            expected = set(METADATA)
            if name.startswith("public"):
                expected.update((*LABELS, "outcome_detail"))
            if set(columns) != expected:
                report["issues"].append(f"{name}: columns differ from course PDF")
            expected_rows = 1300 if name.startswith("public") else 900
            if len(rows) != expected_rows:
                report["issues"].append(f"{name}: expected {expected_rows} rows, got {len(rows)}")
            for c in ("source_id", "preferred_foot", "gender", "resolution", "fps"):
                report["tables"][name][f"{c}_distribution"] = dict(Counter(r.get(c) for r in rows))
        for target, allowed in LABELS.items():
            if name == "public_ground_truth.csv" and target in columns:
                distribution = Counter(r[target] for r in rows)
                report["tables"][name][target] = dict(distribution)
                if set(distribution) - set(allowed):
                    report["issues"].append(f"Unknown labels in {target}")
    public = tables.get("public_ground_truth.csv", [])
    private = tables.get("private_metadata.csv", [])
    if public and private:
        report["shared_source_ids"] = sorted({r.get("source_id", "") for r in public} &
                                              {r.get("source_id", "") for r in private})
        report["shared_clip_ids"] = sorted({r.get("clip") for r in public} & {r.get("clip") for r in private})
    template = tables.get("submission_template.csv", [])
    if template and private and {r.get("clip") for r in template} != {r.get("clip") for r in private}:
        report["issues"].append("Template/private metadata IDs differ")
    for row in public:
        expected_outcome = "goal" if row.get("outcome_detail") == "goal" else "no_goal"
        if row.get("outcome_detail") not in ("goal", "save", "out", "post") or row.get("goal_outcome") != expected_outcome:
            report["issues"].append(f"Outcome mapping mismatch: {row.get('clip')}")
    videos = defaultdict(list)
    for p in files:
        if p.suffix.lower() == ".mp4":
            videos[p.name].append(p)
    hashes = defaultdict(list)
    for partition, rows in (("public", public), ("private", private)):
        listed = {r.get("clip") for r in rows}
        for row in rows:
            clip = row.get("clip", "")
            matches = [p for p in videos[clip] if partition in p.parts]
            if len(matches) != 1:
                report["issues"].append(f"{partition}/{clip}: expected one video, found {len(matches)}")
                continue
            path = matches[0]
            if hash_videos:
                hashes[checksum(path)].append(str(path.relative_to(raw_dir)))
            if decode_videos:
                try:
                    import av
                    with av.open(str(path)) as container:
                        stream = container.streams.video[0]
                        count = sum(1 for _ in container.decode(video=0))
                        info = {"clip": clip, "width": stream.width, "height": stream.height,
                                "fps": float(stream.average_rate) if stream.average_rate else None,
                                "decoded_frames": count, "has_audio": bool(container.streams.audio)}
                        report["video_checks"].append(info)
                        if not count or info["has_audio"]:
                            report["issues"].append(f"{clip}: empty video or unexpected audio")
                        if str(row.get("resolution", "")).replace("×", "x").replace(" ", "") != f'{stream.width}x{stream.height}':
                            report["issues"].append(f"{clip}: resolution metadata differs from decoded video")
                        try:
                            if info["fps"] is None or abs(float(row["fps"]) - info["fps"]) > .1:
                                report["issues"].append(f"{clip}: FPS metadata mismatch")
                        except (ValueError, KeyError):
                            report["issues"].append(f"{clip}: invalid FPS metadata")
                except Exception as exc:
                    report["issues"].append(f"{clip}: decode failed ({type(exc).__name__}: {exc})")
        extras = [str(p.relative_to(raw_dir)) for paths in videos.values() for p in paths
                  if partition in p.parts and p.name not in listed]
        report[f"{partition}_unlisted_videos"] = extras
    report["exact_duplicate_video_groups"] = [paths for paths in hashes.values() if len(paths) > 1]
    report["limitations"] = ["Hashing finds byte-identical duplicates only, not re-encodes.",
                              "Video decoding does not verify the annotated contact cutoff.",
                              "Visual near-duplicates and missing click values require review after access."]
    return report


def save_public_previews(raw_dir, report):
    if not report["tables"].get("public_ground_truth.csv"):
        return
    os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "outputs/.matplotlib"))
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    figure_dir = ROOT / "outputs/figures"
    figure_dir.mkdir(parents=True, exist_ok=True)
    public = report["tables"]["public_ground_truth.csv"]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, target in zip(axes, LABELS):
        counts = public.get(target, {})
        ax.bar(counts.keys(), counts.values())
        ax.set_title(target)
        ax.tick_params(axis="x", rotation=25)
    fig.suptitle("Measured public label counts")
    fig.tight_layout()
    fig.savefig(figure_dir / "public_label_counts.png")
    plt.close(fig)
    # First decoded frame from at most three PUBLIC clips; never annotate private labels.
    import av
    paths = [p for p in sorted(Path(raw_dir).rglob("*.mp4")) if "public" in p.parts][:3]
    if paths:
        fig, axes = plt.subplots(1, len(paths), figsize=(5 * len(paths), 4), squeeze=False)
        for ax, path in zip(axes[0], paths):
            try:
                with av.open(str(path)) as container:
                    frame = next(container.decode(video=0))
                    ax.imshow(frame.to_image())
                ax.set_title(path.name)
            except Exception as exc:
                ax.set_title(f"Decode failed: {type(exc).__name__}")
            ax.axis("off")
        fig.tight_layout()
        fig.savefig(figure_dir / "public_sample_frames.png")
        plt.close(fig)

In [7]:
DECODE_VIDEOS = False
HASH_VIDEOS = False

audit_report = audit_data(DATA_DIR, decode_videos=DECODE_VIDEOS, hash_videos=HASH_VIDEOS)
(OUTPUT_DIR / "data_audit.json").write_text(json.dumps(audit_report, indent=2) + "\n")
save_public_previews(DATA_DIR, audit_report)
print("Audit status:", audit_report["status"])
print("Accessible raw files:", len(audit_report["inventory"]))
for issue in audit_report["issues"]:
    print("-", issue)

Audit status: blocked_no_data
Accessible raw files: 0
- public_ground_truth.csv: expected one file, found 0
- private_metadata.csv: expected one file, found 0
- public_position_clicks.csv: expected one file, found 0
- submission_template.csv: expected one file, found 0


## Counts reported in the assignment

Source: `Project3_CV_2026_v5.pdf`, p. 12. These are not measurements from local CSVs. Majority-class public reference is not the final private educated-guess threshold.

In [8]:
reported_counts = {
    "kick_direction": {"left": 607, "middle": 181, "right": 512},
    "goalkeeper_direction": {"left": 669, "stands in the middle": 63, "right": 568},
    "goal_outcome": {"goal": 939, "no_goal": 361},
}
assert all(sum(counts.values()) == 1300 for counts in reported_counts.values())
print(pd.DataFrame([(target, label, count, round(count / 1300, 4))
                    for target, counts in reported_counts.items()
                    for label, count in counts.items()],
                   columns=["target", "label", "reported_count", "reported_fraction"]).to_string(index=False))

              target                label  reported_count  reported_fraction
      kick_direction                 left             607             0.4669
      kick_direction               middle             181             0.1392
      kick_direction                right             512             0.3938
goalkeeper_direction                 left             669             0.5146
goalkeeper_direction stands in the middle              63             0.0485
goalkeeper_direction                right             568             0.4369
        goal_outcome                 goal             939             0.7223
        goal_outcome              no_goal             361             0.2777


## Actual public metadata, if available

Inspect public records only. Private metadata must not contain any target labels. Source IDs define validation groups, not random clip-level splits.

In [9]:
paths = list((ROOT / "Data").rglob("public_ground_truth.csv"))
if len(paths) == 1:
    public = pd.read_csv(paths[0], dtype={"clip": "string", "source_id": "string"})
    print("Measured public rows:", len(public))
    print("Columns:", public.columns.tolist())
    print("Missing values:\n", public.isna().sum().to_string())
    print("Source groups:", public["source_id"].nunique())
else:
    print("Public CSV unavailable or ambiguous; no data statistics inferred.")

Public CSV unavailable or ambiguous; no data statistics inferred.


## Lightweight validation checks

These checks use temporary synthetic fixtures only, not competition predictions or guessed official IDs. They verify metric alignment/weights, valid CSV handling, and rejection of invalid class strings.

In [10]:
actual = [{"clip": "synthetic_a.mp4", "kick_direction": "left",
           "goalkeeper_direction": "right", "goal_outcome": "goal"},
          {"clip": "synthetic_b.mp4", "kick_direction": "left",
           "goalkeeper_direction": "right", "goal_outcome": "goal"}]
synthetic_predictions = [dict(actual[1], goal_outcome="no_goal"),
                         dict(actual[0], kick_direction="right", goal_outcome="no_goal")]
assert weighted_accuracy(actual, synthetic_predictions)["score"] == 0.55

with tempfile.TemporaryDirectory() as temp:
    temp = Path(temp)
    template, metadata, candidate = [temp / name for name in
                                      ("template.csv", "metadata.csv", "candidate.csv")]
    template.write_text(",".join(COLUMNS) + "\nsynthetic_a.mp4,,,\nsynthetic_b.mp4,,,\n")
    metadata.write_text("clip,source_id\nsynthetic_a.mp4,synthetic_source\nsynthetic_b.mp4,synthetic_source\n")
    with candidate.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=COLUMNS)
        writer.writeheader()
        writer.writerows(actual)
    assert validate_submission(candidate, template, metadata, expected_rows=2)["valid"]
    candidate.write_text(candidate.read_text().replace(",left,", ",invalid_label,"))
    try:
        validate_submission(candidate, template, metadata, expected_rows=2)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid labels must be rejected")
print("Synthetic metric and submission checks passed.")

Synthetic metric and submission checks passed.


## Validate a future competition submission

No submission is generated here. Leave `SUBMISSION_PATH = None` until a real candidate exists. The validator requires the actual official template and metadata; it never invents IDs or columns.

In [11]:
SUBMISSION_PATH = None

if SUBMISSION_PATH is not None:
    templates = list(DATA_DIR.rglob("submission_template.csv"))
    metadata_files = list(DATA_DIR.rglob("private_metadata.csv"))
    if len(templates) != 1 or len(metadata_files) != 1:
        raise FileNotFoundError("Exactly one official template and private metadata file are required")
    print(validate_submission(Path(SUBMISSION_PATH), templates[0], metadata_files[0]))
else:
    print("No competition candidate selected; official-template verification is pending data access.")

No competition candidate selected; official-template verification is pending data access.


## Next phases

After restoring data access, rerun the download and audit cells, inspect the packaged README/scorer/template, review public previews and source overlap, and record measured findings in `AGENTS.md`. Stop before baseline fitting until Phase 1 is authorized. Future preprocessing, training, validation, inference, and CSV generation must be implemented here, with no external solution `.py` files.